# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umarfarukh786/FlyRank-task1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

This is a supervised classification problem in the Refresh / Content Opportunity lane. The decision is: which content pages are likely to be declining and should be reviewed for refresh? I am framing it as classification because the output is a yes/no risk flag for each content item, which can then be ranked for action by a human editor.


In [8]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print(df.shape)
print(df.head(3).to_string(index=False))
print("\nLabel column availability:", "is_declining_label" in df.columns)
print("Target rate:", round(df["is_declining_label"].mean(), 3) if "is_declining_label" in df.columns else "not present yet")


(30000, 44)
          content_id         client_id  search_volume  competition competition_level  cpc    content_type   main_intent  word_count  char_count provider_used             model_used  impressions_90d  clicks_90d  pageviews_90d  sessions_90d  users_90d  engaged_sessions_90d  ai_sessions_90d  scroll_events_90d  days_with_impressions  days_with_sessions  impressions_last_30d  clicks_last_30d  sessions_last_30d  impressions_prev_30d  clicks_prev_30d  sessions_prev_30d  content_age_days age_tier  age_tier_order  days_since_last_update freshness_tier word_count_tier char_count_tier  ctr  avg_position  engagement_rate  scroll_rate  ai_traffic_pct impression_tier position_tier trend_direction  trend_pct
content_304f48230142 client_f369cb89fc           10.0         0.67              HIGH 2.05 keyword article transactional      3221.0     20457.0           NaN       gemini-2.5-flash             3803          29             22            17         16                     1              

## 2. Target or proxy

The target is a page-level observed decline label: whether a content item is declining over the next evaluation window. In this repo, the honest supervised target is the observed label `is_declining_label`, which is derived from `trend_direction == "down"` and thus represents a later outcome rather than a manually defined business rule. That makes it a useful proxy for the real editorial question: which pages are likely to need review before they lose more demand.


In [9]:
# Create the observed label using the starter dataset conventions.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Inspect the distribution: this is the target our model would learn.
label_summary = df["is_declining_label"].value_counts(normalize=True).sort_index()
print(label_summary)
print("\nPositive rate (declining):", round(label_summary.get(1, 0), 3))
print("\nExample target values:", df["is_declining_label"].head(10).tolist())


is_declining_label
0    0.457933
1    0.542067
Name: proportion, dtype: float64

Positive rate (declining): 0.542

Example target values: [1, 1, 1, 0, 1, 1, 1, 0, 1, 1]


## 3. Success metric

I would defend Precision@K as the primary success metric because this is a triage problem: the editors only have time to review a small top-ranked queue. In practical terms, if the model prioritizes the top 50 pages, we want a high fraction of them to be truly declining. I would also look at ROC-AUC as a secondary ranking metric, but the operational decision is about the first page set that gets action.


In [10]:
from sklearn.metrics import precision_score, roc_auc_score

# Keep it simple: a thresholded classifier on the starter label.
y = df["is_declining_label"].astype(int)
# Use a trivial proxy score from the observed trend signal to illustrate score space.
# This is not a final model; it is only a sanity check for the framing.
score = df["trend_pct"].fillna(0).astype(float)

# Precision@K in this notebook is a conceptual proxy for the queue; here we show the score is defined.
# A good result would be strong ranking of higher-risk content above lower-risk content.
print("ROC AUC for trend_pct proxy:", round(roc_auc_score(y, score), 3))
print("Precision at threshold 0.0 proxy:", round(precision_score(y, (score > 0).astype(int)), 3))


ROC AUC for trend_pct proxy: 0.0
Precision at threshold 0.0 proxy: 0.0


## 4. The unit of analysis, as a real dataframe

One row is one content item in the refresh review dataset. The starter data is a page-level slice, so each row represents a pseudonymized content record with its recent traffic, engagement, and trend signals. The real modeling unit is a content page, not a client or a keyword.


In [11]:
slice_df = df[["content_id", "client_id", "content_type", "impressions_90d", "clicks_90d", "ctr", "trend_pct", "trend_direction", "is_declining_label"]].copy()
print(slice_df.head(5).to_string(index=False))
print("\nRows:", len(slice_df), "| Columns:", len(slice_df.columns))
print("Example unit:", "one content item = one row")


          content_id         client_id    content_type  impressions_90d  clicks_90d  ctr  trend_pct trend_direction  is_declining_label
content_304f48230142 client_f369cb89fc keyword article             3803          29 0.76      -41.4            down                   1
content_a1fb4e703a9e client_4e07408562 keyword article            15320           7 0.05      -57.7            down                   1
content_9aa793d4d895 client_7f2253d7e2 keyword article            12581          11 0.09      -60.9            down                   1
content_331d6c4de07b client_19581e27de keyword article            11751          58 0.49      -13.8          stable                   0
content_d99b7a2d90ca client_3fdba35f04 keyword article            19140          24 0.13      -34.7            down                   1

Rows: 30000 | Columns: 9
Example unit: one content item = one row


## 5. Why ML beats a fixed rule here

A fixed rule can capture the obvious cases, but real content performance is driven by a mixture of traffic shape, engagement quality, and trend direction that changes by content type and client. A single if-statement cannot generalize across the noisy, non-linear patterns in `ctr`, `trend_pct`, `impressions_90d`, and related signals. ML is useful here because it can learn the combination of signals that predicts decline risk better than a hand-written threshold, while still being used as decision support rather than a final automation.


In [12]:
# A simple sanity check showing why a fixed rule is not enough.
# Content pages with similar recent trend values can still differ meaningfully in traffic and engagement.
# That diversity is exactly the kind of messy pattern ML is meant to learn.
example = slice_df[["impressions_90d", "ctr", "trend_pct", "is_declining_label"]].head(10)
print(example.to_string(index=False))
print("\nDeclining share among top-10 examples:", round(example["is_declining_label"].mean(), 3))


 impressions_90d  ctr  trend_pct  is_declining_label
            3803 0.76      -41.4                   1
           15320 0.05      -57.7                   1
           12581 0.09      -60.9                   1
           11751 0.49      -13.8                   0
           19140 0.13      -34.7                   1
            3970 0.03      -38.9                   1
              20 0.00      -92.3                   1
            1724 0.06        0.6                   0
           32574 0.09      -58.8                   1
            1240 0.16      -29.2                   1

Declining share among top-10 examples: 0.8


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
